# Exécution locale
Choisir le noyau `.venv`, exécuter la cellule des chemins, installer les dépendances puis charger `.env`.
Pour commencer, utiliser la section **Proxy Server**. Cohere, Groq, Langfuse et LangChain sont optionnels : ne pas exécuter tout le cours automatiquement.


In [ ]:
# Trouver la racine du projet depuis notebooks/ ou depuis la racine.
from pathlib import Path

PROJECT_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "config" / "llm.yaml").is_file()
     and (folder / "requirements.txt").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Ouvre ce notebook dans le dossier LLM-proxies.")
CONFIG_DIR = PROJECT_ROOT / "config"
LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)
print("Configuration :", CONFIG_DIR)


In [ ]:
%pip install -r "{PROJECT_ROOT / 'requirements.txt'}"


In [ ]:
# Charger les clés depuis le fichier .env à la racine du projet.
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)

def require_keys(*names):
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise ValueError("Renseigne ces variables dans .env : " + ", ".join(missing))

require_keys("OPENAI_API_KEY", "LITELLM_MASTER_KEY")
print("Configuration locale chargée (clés non affichées).")


In [ ]:
from litellm import completion, acompletion
from pprint import pprint

messages = [
    {
        "role": "user", # system, assistant
        "content": "لماذا تبدو السماء زرقاء بالنهار؟"
    }
]

response = completion(
        #model="openai/gpt-4o-mini",
        model="openai/gpt-4o-mini",
        messages=messages,
        temperature=0.5,
        max_tokens=200
    )

In [ ]:
response.model

In [ ]:
pprint(response.choices[0].message.content)

### SDK Logging

In [ ]:
import litellm
import os
import json

logs_dir = str(LOG_DIR / "sdk")
os.makedirs(logs_dir, exist_ok=True)

def log_success(kwargs, completion_obj, start_time, end_time):
    with open(f"{logs_dir}/success-logs.jsonl", "a") as dest:
        dest.write(
            json.dumps({
                "kwargs": kwargs,
                "completion_obj": completion_obj,
                "start_time": start_time,
                "end_time": end_time,
            }, ensure_ascii=False, default=str ) + "\n"
        )

def log_failure(kwargs, completion_obj, start_time, end_time):
    with open(f"{logs_dir}/failure-logs.jsonl", "a") as dest:
        dest.write(
            json.dumps({
                "kwargs": kwargs,
                "completion_obj": completion_obj,
                "start_time": start_time,
                "end_time": end_time,
            }, ensure_ascii=False, default=str ) + "\n"
        )

litellm.success_callback = [log_success]
litellm.failure_callback = [log_failure]

In [ ]:
from litellm import completion

messages = [
    {
        "role": "user", # system, assistant
        "content": "لماذا تبدو السماء زرقاء بالنهار؟"
    }
]

response = completion(
            model="openai/gpt-4o-mini",
            messages=messages,
            temperature=0.5,
            max_tokens=200
        )

### Proxy Server

In [ ]:
%pip install -r "{PROJECT_ROOT / 'requirements.txt'}"


In [ ]:
# ============ check any litellm processes
# !pgrep -fl litellm

# ============ kill any litellm processes
# !pkill -f litellm

In [ ]:
# Charger les clés depuis le fichier .env à la racine du projet.
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)

def require_keys(*names):
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise ValueError("Renseigne ces variables dans .env : " + ", ".join(missing))

require_keys("OPENAI_API_KEY", "LITELLM_MASTER_KEY")
print("Configuration locale chargée (clés non affichées).")


In [ ]:
# La configuration est conservée dans config/llm.yaml.
config_path = CONFIG_DIR / "llm.yaml"
print(config_path.read_text())


In [ ]:
# Ce bloc démarre LiteLLM dans un processus séparé, puis vérifie sa disponibilité.
# Il nécessite les cellules précédentes : chemins du projet et chargement de .env.
import subprocess  # Démarrer et contrôler un programme externe.
import time        # Faire une pause entre les vérifications du serveur.
import os          # Lire et copier les variables d'environnement, dont les clés.
import sys         # Retrouver l'interpréteur du noyau Python sélectionné.
from pathlib import Path  # Construire des chemins de fichiers.
import httpx       # Envoyer une requête HTTP de vérification au proxy.

# Arrêter ici avec un message explicite si une clé obligatoire est absente.
# Cette fonction est définie dans la cellule qui charge le fichier .env.
require_keys("OPENAI_API_KEY", "LITELLM_MASTER_KEY")

# globals() contient les variables déjà définies dans le notebook.
# get() renvoie None si aucun proxy_process n'a encore été créé : cela évite
# une erreur lors de la première exécution de cette cellule.
# Conserver un processus par configuration : changer de section ne doit pas arrêter le LB.
proxy_processes = globals().setdefault("proxy_processes", {})
previous_proxy = proxy_processes.get("llm.yaml")

# poll() renvoie None si le processus fonctionne encore ; sinon, son code de sortie.
# Ne pas lancer un second proxy sur le même port quand le précédent est actif.
# Ce contrôle concerne uniquement le processus conservé par ce notebook.
if previous_proxy is not None and previous_proxy.poll() is None:
    previous_proxy.terminate()  # Demander au serveur de s'arrêter.
    previous_proxy.wait(timeout=10)  # Attendre sa fin avant de réutiliser le port.
    # Si le serveur ne s'arrête pas en 10 secondes, wait() lève TimeoutExpired.

# Le nouveau processus hérite des clés et des autres variables du notebook.
# copy() permet de modifier sa configuration sans changer os.environ lui-même.
# LiteLLM lit DEBUG et DETAILED_DEBUG comme des booléens : "release" est invalide.
proxy_env = os.environ.copy()
proxy_env["DEBUG"] = "false"
proxy_env["DETAILED_DEBUG"] = "false"

# sys.executable est le chemin du Python utilisé par le noyau du notebook.
# with_name("litellm") vise l'exécutable voisin installé dans ce même environnement.
# Cela suppose que LiteLLM a été installé dans l'environnement du noyau sélectionné.
litellm_executable = Path(sys.executable).with_name("litellm")
config_path = CONFIG_DIR / "llm.yaml"
log_path = LOG_DIR / "proxy.log"

# Chaque élément de la liste est un argument de la commande, sans passer par un shell.
# Équivalent : litellm --port 4000 --config <chemin du YAML>
command = [str(litellm_executable), "--port", "4000", "--config", str(config_path)]

# "w" crée le journal ou remplace son contenu précédent.
# Le bloc with ferme le fichier côté notebook après le lancement ; le processus
# enfant conserve sa propre référence au fichier pour continuer à écrire dedans.
with log_path.open("w") as log_file:
    # Popen lance le programme et rend immédiatement la main au notebook.
    # Il crée un processus séparé : ce n'est pas un thread Python.
    proxy_process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,  # Dossier de travail du serveur ; contient notamment .env.
        env=proxy_env,     # Variables d'environnement transmises au serveur.
        stdout=log_file,   # Écrire les messages ordinaires dans proxy.log.
        stderr=subprocess.STDOUT,  # Écrire aussi les erreurs dans le même journal.
    )

proxy_processes["llm.yaml"] = proxy_process

# Le lancement d'un processus ne garantit pas que le serveur est déjà prêt.
# Faire au maximum 60 essais ; chacun peut prendre jusqu'à 2 secondes pour
# la requête HTTP, puis 1 seconde de pause. Ce n'est donc pas un délai strict de 60 s.
for attempt in range(60):
    # Si le programme s'est arrêté, afficher la fin de son journal pour diagnostiquer.
    # [-6000:] conserve les 6000 derniers caractères, pas les 6000 dernières lignes.
    if proxy_process.poll() is not None:
        raise RuntimeError(log_path.read_text()[-6000:])

    try:
        # GET /v1/models demande les modèles exposés par le proxy.
        # Cette vérification ne génère pas de réponse avec un modèle fournisseur.
        status = httpx.get(
            "http://127.0.0.1:4000/v1/models",
            # Bearer est le schéma d'authentification ; la valeur est la clé maître.
            headers={"Authorization": "Bearer " + os.environ["LITELLM_MASTER_KEY"]},
            timeout=2,  # Délai des opérations réseau de cette vérification.
        )
        # HTTP 200 signifie que cette route répond avec succès et accepte la clé.
        # Cela ne prouve pas encore que chaque modèle fournisseur est utilisable.
        if status.status_code == 200:
            print("Proxy prêt sur http://127.0.0.1:4000/v1")
            break  # Quitter la boucle dès que la vérification réussit.

    except httpx.RequestError:
        # Pendant le démarrage, le port peut ne pas encore accepter de connexion.
        # Ignorer cet échec réseau provisoire et réessayer après la pause.
        # Une réponse HTTP 401/500 ne passe pas ici : elle échoue au test du code 200.
        pass

    time.sleep(1)  # Éviter d'envoyer les vérifications en continu.

else:
    # Le else appartient au for : il s'exécute si les 60 essais se terminent sans break.
    # Le processus peut encore fonctionner ; inspecter son journal avant de réessayer.
    raise RuntimeError("Proxy non disponible. Consulte logs/proxy.log.")


In [ ]:
import os
import openai

# Créer un client OpenAI qui envoie les requêtes au proxy LiteLLM.
client = openai.OpenAI(
    # Utiliser la même clé maître que celle définie au démarrage du proxy.
    # Cette clé authentifie le client auprès de LiteLLM.
    api_key=os.environ["LITELLM_MASTER_KEY"],

    # Adresse du proxy dans l'environnement Colab :
    # 127.0.0.1 = machine locale, 4000 = port, /v1 = API compatible OpenAI.
    base_url="http://127.0.0.1:4000/v1",
)

# Préparer le message à envoyer au modèle.
messages = [
    {
        # "user" indique que le message vient de l'utilisateur.
        "role": "user",

        # Question : « Pourquoi le ciel paraît-il bleu pendant la journée ? »
        "content": "لماذا تبدو السماء زرقاء بالنهار؟",
    }
]

# Envoyer la requête au proxy, qui la transmet au fournisseur du modèle.
response = client.chat.completions.create(
    # Alias du modèle déclaré dans le fichier llm.yaml.
    model="openai-gpt4o-mini",

    # Messages constituant la conversation.
    messages=messages,
)

# Afficher le texte de la première réponse du modèle.
print(response.choices[0].message.content)

### Load Balancer

In [ ]:
# La configuration est conservée dans config/llm-lb.yaml.
config_path = CONFIG_DIR / "llm-lb.yaml"
print(config_path.read_text())


In [ ]:
# Ce bloc démarre LiteLLM dans un processus séparé, puis vérifie sa disponibilité.
# Il nécessite les cellules précédentes : chemins du projet et chargement de .env.
import subprocess  # Démarrer et contrôler un programme externe.
import time        # Faire une pause entre les vérifications du serveur.
import os          # Lire et copier les variables d'environnement, dont les clés.
import sys         # Retrouver l'interpréteur du noyau Python sélectionné.
from pathlib import Path  # Construire des chemins de fichiers.
import httpx       # Envoyer une requête HTTP de vérification au proxy.

# Arrêter ici avec un message explicite si une clé obligatoire est absente.
# Cette fonction est définie dans la cellule qui charge le fichier .env.
require_keys("OPENAI_API_KEY", "GROQ_API_KEY", "LITELLM_MASTER_KEY")

# globals() contient les variables déjà définies dans le notebook.
# get() renvoie None si aucun proxy_process n'a encore été créé : cela évite
# une erreur lors de la première exécution de cette cellule.
# Conserver un processus par configuration : changer de section ne doit pas arrêter le LB.
proxy_processes = globals().setdefault("proxy_processes", {})
previous_proxy = proxy_processes.get("llm-lb.yaml")

# poll() renvoie None si le processus fonctionne encore ; sinon, son code de sortie.
# Ne pas lancer un second proxy sur le même port quand le précédent est actif.
# Ce contrôle concerne uniquement le processus conservé par ce notebook.
if previous_proxy is not None and previous_proxy.poll() is None:
    previous_proxy.terminate()  # Demander au serveur de s'arrêter.
    previous_proxy.wait(timeout=10)  # Attendre sa fin avant de réutiliser le port.
    # Si le serveur ne s'arrête pas en 10 secondes, wait() lève TimeoutExpired.

# Le nouveau processus hérite des clés et des autres variables du notebook.
# copy() permet de modifier sa configuration sans changer os.environ lui-même.
# LiteLLM lit DEBUG et DETAILED_DEBUG comme des booléens : "release" est invalide.
proxy_env = os.environ.copy()
proxy_env["DEBUG"] = "false"
proxy_env["DETAILED_DEBUG"] = "false"

# sys.executable est le chemin du Python utilisé par le noyau du notebook.
# with_name("litellm") vise l'exécutable voisin installé dans ce même environnement.
# Cela suppose que LiteLLM a été installé dans l'environnement du noyau sélectionné.
litellm_executable = Path(sys.executable).with_name("litellm")
config_path = CONFIG_DIR / "llm-lb.yaml"
log_path = LOG_DIR / "proxy.log"

# Chaque élément de la liste est un argument de la commande, sans passer par un shell.
# Équivalent : litellm --port 4001 --config <chemin du YAML>
command = [str(litellm_executable), "--port", "4001", "--config", str(config_path)]

# "w" crée le journal ou remplace son contenu précédent.
# Le bloc with ferme le fichier côté notebook après le lancement ; le processus
# enfant conserve sa propre référence au fichier pour continuer à écrire dedans.
# Réserver un port distinct du proxy simple pour éviter de contacter l'ancien serveur.
# Vérifier qu'il est libre avant de lancer LiteLLM.
import socket
# SO_REUSEADDR évite le faux conflit causé par les connexions en TIME_WAIT.
with socket.socket() as probe:
    probe.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        probe.bind(("127.0.0.1", 4001))
    except OSError as exc:
        raise RuntimeError("Port 4001 occupé : arrête le précédent proxy load balancing.") from exc

with log_path.open("w") as log_file:
    # Popen lance le programme et rend immédiatement la main au notebook.
    # Il crée un processus séparé : ce n'est pas un thread Python.
    proxy_process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,  # Dossier de travail du serveur ; contient notamment .env.
        env=proxy_env,     # Variables d'environnement transmises au serveur.
        stdout=log_file,   # Écrire les messages ordinaires dans proxy.log.
        stderr=subprocess.STDOUT,  # Écrire aussi les erreurs dans le même journal.
    )

proxy_processes["llm-lb.yaml"] = proxy_process

# Le lancement d'un processus ne garantit pas que le serveur est déjà prêt.
# Faire au maximum 60 essais ; chacun peut prendre jusqu'à 2 secondes pour
# la requête HTTP, puis 1 seconde de pause. Ce n'est donc pas un délai strict de 60 s.
for attempt in range(60):
    # Si le programme s'est arrêté, afficher la fin de son journal pour diagnostiquer.
    # [-6000:] conserve les 6000 derniers caractères, pas les 6000 dernières lignes.
    if proxy_process.poll() is not None:
        raise RuntimeError(log_path.read_text()[-6000:])

    try:
        # GET /v1/models demande les modèles exposés par le proxy.
        # Cette vérification ne génère pas de réponse avec un modèle fournisseur.
        status = httpx.get(
            "http://127.0.0.1:4001/v1/models",
            # Bearer est le schéma d'authentification ; la valeur est la clé maître.
            headers={"Authorization": "Bearer " + os.environ["LITELLM_MASTER_KEY"]},
            timeout=2,  # Délai des opérations réseau de cette vérification.
        )
        # HTTP 200 signifie que cette route répond avec succès et accepte la clé.
        # Cela ne prouve pas encore que chaque modèle fournisseur est utilisable.
        if status.status_code == 200:
            # Vérifier que le serveur actif expose bien l'alias de cette section.
            available_models = {item["id"] for item in status.json().get("data", [])}
            if "global-llm" not in available_models:
                raise RuntimeError(
                    "Le serveur sur le port 4001 ne propose pas global-llm. "
                    "Arrête l'ancien proxy, puis relance cette cellule avec llm-lb.yaml. "
                    "Alias disponibles : " + ", ".join(sorted(available_models))
                )
            print("Proxy prêt sur http://127.0.0.1:4001/v1")
            break  # Quitter la boucle dès que la vérification réussit.

    except httpx.RequestError:
        # Pendant le démarrage, le port peut ne pas encore accepter de connexion.
        # Ignorer cet échec réseau provisoire et réessayer après la pause.
        # Une réponse HTTP 401/500 ne passe pas ici : elle échoue au test du code 200.
        pass

    time.sleep(1)  # Éviter d'envoyer les vérifications en continu.

else:
    # Le else appartient au for : il s'exécute si les 60 essais se terminent sans break.
    # Le processus peut encore fonctionner ; inspecter son journal avant de réessayer.
    raise RuntimeError("Proxy non disponible. Consulte logs/proxy.log.")


In [ ]:
import openai
from pprint import pprint

client = openai.OpenAI(
    api_key=os.environ["LITELLM_MASTER_KEY"],
    base_url="http://127.0.0.1:4001/v1"
)

In [ ]:
# Le client doit pointer sur le serveur lancé avec config/llm-lb.yaml.
# Vérifier l'alias avant d'envoyer une requête au fournisseur.
available_models = {item.id for item in client.models.list().data}
if "global-llm" not in available_models:
    raise RuntimeError(
        "global-llm absent du proxy actif. Relance le bloc de démarrage Load Balancer. "
        "Alias disponibles : " + ", ".join(sorted(available_models))
    )

messages = [
    {
        "role": "user", # system, assistant
        "content": "لماذا تبدو السماء زرقاء بالنهار؟"
    }
]

# Conserver les en-têtes HTTP pour identifier le déploiement sélectionné.
raw_response = client.chat.completions.with_raw_response.create(
    model="global-llm",
    messages=messages,
)
# Convertir le corps JSON en objet de réponse habituel.
response = raw_response.parse()


In [ ]:
# Recréer le client LB : les autres sections peuvent avoir remplacé la variable client.
import os
import openai
import httpx

client = openai.OpenAI(
    api_key=os.environ["LITELLM_MASTER_KEY"],
    base_url="http://127.0.0.1:4001/v1",
    max_retries=0,  # Le test compte les appels explicites, sans retries automatiques du SDK.
)
try:
    aliases = {model.id for model in client.models.list().data}
except openai.APIConnectionError as exc:
    raise RuntimeError(
        "Proxy Load Balancer arrêté sur 4001. Exécute d'abord sa cellule de démarrage."
    ) from exc
if "global-llm" not in aliases:
    raise RuntimeError("Le proxy actif n'expose pas global-llm : vérifie llm-lb.yaml.")

from collections import Counter
import time

counts = Counter()

# Chaque tour envoie une nouvelle requête au proxy.
for i in range(10):
    try:
        raw = client.chat.completions.with_raw_response.create(
            model="global-llm",
            messages=[
                {"role": "user", "content": "Réponds uniquement : Bonjour"}
            ],
            max_tokens=30,
        )

        selected_model = raw.headers.get(
            "x-litellm-model-name", "Modèle non indiqué"
        )
        deployment_id = raw.headers.get(
            "x-litellm-model-id", "ID non indiqué"
        )

        counts[selected_model] += 1
        print(f"Appel {i + 1} : {selected_model}")
        print(f"  Déploiement : {deployment_id}")

    except openai.APIConnectionError as error:
        print("Connexion perdue avec le proxy. Arrêt du test :", error)
        break
    except Exception as error:
        print(f"Appel {i + 1} échoué : {error}")

    time.sleep(2)

print("\nRépartition des appels réussis :")
for model, count in counts.items():
    print(f"{model} : {count} appel(s)")

In [ ]:
# response.model peut conserver l'alias public demandé : global-llm.
print("Alias dans la réponse :", response.model)
# Ces en-têtes sont fournis par le proxy LiteLLM installé dans ce projet.
print("Modèle sélectionné :", raw_response.headers.get("x-litellm-model-name", "Non fourni"))
print("ID du déploiement :", raw_response.headers.get("x-litellm-model-id", "Non fourni"))
print("Réponse :", response.choices[0].message.content)


In [ ]:
pprint(
    response.choices[0].message.content
)

### Fallbacks

In [ ]:
# Charger les clés depuis le fichier .env à la racine du projet.
import os
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env", override=True)

def require_keys(*names):
    missing = [name for name in names if not os.getenv(name)]
    if missing:
        raise ValueError("Renseigne ces variables dans .env : " + ", ".join(missing))

require_keys("OPENAI_API_KEY", "LITELLM_MASTER_KEY")
print("Configuration locale chargée (clés non affichées).")
require_keys("GROQ_API_KEY")


In [ ]:
# La configuration est conservée dans config/llm-fallback.yaml.
config_path = CONFIG_DIR / "llm-fallback.yaml"
print(config_path.read_text())


In [ ]:
# Ce bloc démarre LiteLLM dans un processus séparé, puis vérifie sa disponibilité.
# Il nécessite les cellules précédentes : chemins du projet et chargement de .env.
import subprocess  # Démarrer et contrôler un programme externe.
import time        # Faire une pause entre les vérifications du serveur.
import os          # Lire et copier les variables d'environnement, dont les clés.
import sys         # Retrouver l'interpréteur du noyau Python sélectionné.
from pathlib import Path  # Construire des chemins de fichiers.
import httpx       # Envoyer une requête HTTP de vérification au proxy.

# Arrêter ici avec un message explicite si une clé obligatoire est absente.
# Cette fonction est définie dans la cellule qui charge le fichier .env.
require_keys("OPENAI_API_KEY", "LITELLM_MASTER_KEY")

# globals() contient les variables déjà définies dans le notebook.
# get() renvoie None si aucun proxy_process n'a encore été créé : cela évite
# une erreur lors de la première exécution de cette cellule.
# Conserver un processus par configuration : changer de section ne doit pas arrêter le LB.
proxy_processes = globals().setdefault("proxy_processes", {})
previous_proxy = proxy_processes.get("llm-fallback.yaml")

# poll() renvoie None si le processus fonctionne encore ; sinon, son code de sortie.
# Ne pas lancer un second proxy sur le même port quand le précédent est actif.
# Ce contrôle concerne uniquement le processus conservé par ce notebook.
if previous_proxy is not None and previous_proxy.poll() is None:
    previous_proxy.terminate()  # Demander au serveur de s'arrêter.
    previous_proxy.wait(timeout=10)  # Attendre sa fin avant de réutiliser le port.
    # Si le serveur ne s'arrête pas en 10 secondes, wait() lève TimeoutExpired.

# Le nouveau processus hérite des clés et des autres variables du notebook.
# copy() permet de modifier sa configuration sans changer os.environ lui-même.
# LiteLLM lit DEBUG et DETAILED_DEBUG comme des booléens : "release" est invalide.
proxy_env = os.environ.copy()
proxy_env["DEBUG"] = "false"
proxy_env["DETAILED_DEBUG"] = "false"

# sys.executable est le chemin du Python utilisé par le noyau du notebook.
# with_name("litellm") vise l'exécutable voisin installé dans ce même environnement.
# Cela suppose que LiteLLM a été installé dans l'environnement du noyau sélectionné.
litellm_executable = Path(sys.executable).with_name("litellm")
config_path = CONFIG_DIR / "llm-fallback.yaml"
log_path = LOG_DIR / "proxy.log"

# Chaque élément de la liste est un argument de la commande, sans passer par un shell.
# Équivalent : litellm --port 4000 --config <chemin du YAML>
command = [str(litellm_executable), "--port", "4000", "--config", str(config_path)]

# "w" crée le journal ou remplace son contenu précédent.
# Le bloc with ferme le fichier côté notebook après le lancement ; le processus
# enfant conserve sa propre référence au fichier pour continuer à écrire dedans.
with log_path.open("w") as log_file:
    # Popen lance le programme et rend immédiatement la main au notebook.
    # Il crée un processus séparé : ce n'est pas un thread Python.
    proxy_process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,  # Dossier de travail du serveur ; contient notamment .env.
        env=proxy_env,     # Variables d'environnement transmises au serveur.
        stdout=log_file,   # Écrire les messages ordinaires dans proxy.log.
        stderr=subprocess.STDOUT,  # Écrire aussi les erreurs dans le même journal.
    )

proxy_processes["llm-fallback.yaml"] = proxy_process

# Le lancement d'un processus ne garantit pas que le serveur est déjà prêt.
# Faire au maximum 60 essais ; chacun peut prendre jusqu'à 2 secondes pour
# la requête HTTP, puis 1 seconde de pause. Ce n'est donc pas un délai strict de 60 s.
for attempt in range(60):
    # Si le programme s'est arrêté, afficher la fin de son journal pour diagnostiquer.
    # [-6000:] conserve les 6000 derniers caractères, pas les 6000 dernières lignes.
    if proxy_process.poll() is not None:
        raise RuntimeError(log_path.read_text()[-6000:])

    try:
        # GET /v1/models demande les modèles exposés par le proxy.
        # Cette vérification ne génère pas de réponse avec un modèle fournisseur.
        status = httpx.get(
            "http://127.0.0.1:4000/v1/models",
            # Bearer est le schéma d'authentification ; la valeur est la clé maître.
            headers={"Authorization": "Bearer " + os.environ["LITELLM_MASTER_KEY"]},
            timeout=2,  # Délai des opérations réseau de cette vérification.
        )
        # HTTP 200 signifie que cette route répond avec succès et accepte la clé.
        # Cela ne prouve pas encore que chaque modèle fournisseur est utilisable.
        if status.status_code == 200:
            print("Proxy prêt sur http://127.0.0.1:4000/v1")
            break  # Quitter la boucle dès que la vérification réussit.

    except httpx.RequestError:
        # Pendant le démarrage, le port peut ne pas encore accepter de connexion.
        # Ignorer cet échec réseau provisoire et réessayer après la pause.
        # Une réponse HTTP 401/500 ne passe pas ici : elle échoue au test du code 200.
        pass

    time.sleep(1)  # Éviter d'envoyer les vérifications en continu.

else:
    # Le else appartient au for : il s'exécute si les 60 essais se terminent sans break.
    # Le processus peut encore fonctionner ; inspecter son journal avant de réessayer.
    raise RuntimeError("Proxy non disponible. Consulte logs/proxy.log.")


### Observation

In [ ]:
# Charger les secrets sans les afficher. Les clés Langfuse viennent du projet Langfuse.
import os
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env", override=True)

def require_keys(*names):
    missing = [name for name in names if not os.getenv(name, "").strip()]
    if missing:
        raise ValueError("Renseigne ces variables dans .env puis relance cette cellule : " + ", ".join(missing))

# Ce YAML déclare OpenAI et Groq ; Langfuse reçoit les traces du proxy.
require_keys("OPENAI_API_KEY", "GROQ_API_KEY", "LITELLM_MASTER_KEY",
             "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_HOST")
print("Clés nécessaires à l’observation présentes (valeurs non affichées).")


In [ ]:
# Utiliser l’URL du projet Langfuse, avec sa région ou son installation locale.
# Une clé OpenAI/Groq ne remplace pas une clé Langfuse.
from urllib.parse import urlparse
host = urlparse(os.environ.get("LANGFUSE_HOST", ""))
if host.scheme not in ("http", "https") or not host.netloc:
    raise ValueError("LANGFUSE_HOST doit être une URL HTTP(S) du serveur Langfuse.")
print("URL Langfuse valide ; la réception des traces reste à vérifier après un appel.")


In [ ]:
# L’intégration langfuse_otel exporte les traces via OpenTelemetry.
# Elle ne nécessite pas le SDK Python langfuse, contrairement au callback historique.
%pip install opentelemetry-sdk opentelemetry-exporter-otlp-proto-http


In [ ]:
# La configuration est conservée dans config/llm-lanfuse.yaml.
config_path = CONFIG_DIR / "llm-lanfuse.yaml"
print(config_path.read_text())


In [ ]:
# Ce bloc démarre LiteLLM dans un processus séparé, puis vérifie sa disponibilité.
# Il nécessite les cellules précédentes : chemins du projet et chargement de .env.
import subprocess  # Démarrer et contrôler un programme externe.
import time        # Faire une pause entre les vérifications du serveur.
import os          # Lire et copier les variables d'environnement, dont les clés.
import sys         # Retrouver l'interpréteur du noyau Python sélectionné.
from pathlib import Path  # Construire des chemins de fichiers.
import httpx       # Envoyer une requête HTTP de vérification au proxy.

# Arrêter ici avec un message explicite si une clé obligatoire est absente.
# Cette fonction est définie dans la cellule qui charge le fichier .env.
require_keys("OPENAI_API_KEY", "GROQ_API_KEY", "LITELLM_MASTER_KEY",
             "LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY", "LANGFUSE_HOST")

# globals() contient les variables déjà définies dans le notebook.
# get() renvoie None si aucun proxy_process n'a encore été créé : cela évite
# une erreur lors de la première exécution de cette cellule.
# Conserver un processus par configuration : changer de section ne doit pas arrêter le LB.
proxy_processes = globals().setdefault("proxy_processes", {})
previous_proxy = proxy_processes.get("llm-lanfuse.yaml")

# poll() renvoie None si le processus fonctionne encore ; sinon, son code de sortie.
# Ne pas lancer un second proxy sur le même port quand le précédent est actif.
# Ce contrôle concerne uniquement le processus conservé par ce notebook.
if previous_proxy is not None and previous_proxy.poll() is None:
    previous_proxy.terminate()  # Demander au serveur de s'arrêter.
    previous_proxy.wait(timeout=10)  # Attendre sa fin avant de réutiliser le port.
    # Si le serveur ne s'arrête pas en 10 secondes, wait() lève TimeoutExpired.

# Le nouveau processus hérite des clés et des autres variables du notebook.
# copy() permet de modifier sa configuration sans changer os.environ lui-même.
# LiteLLM lit DEBUG et DETAILED_DEBUG comme des booléens : "release" est invalide.
proxy_env = os.environ.copy()
proxy_env["DEBUG"] = "false"
proxy_env["DETAILED_DEBUG"] = "false"

# sys.executable est le chemin du Python utilisé par le noyau du notebook.
# with_name("litellm") vise l'exécutable voisin installé dans ce même environnement.
# Cela suppose que LiteLLM a été installé dans l'environnement du noyau sélectionné.
litellm_executable = Path(sys.executable).with_name("litellm")
config_path = CONFIG_DIR / "llm-lanfuse.yaml"
LOG_DIR.mkdir(parents=True, exist_ok=True)
log_path = LOG_DIR / "observation-proxy.log"

# Chaque élément de la liste est un argument de la commande, sans passer par un shell.
# Équivalent : litellm --port 4000 --config <chemin du YAML>
command = [str(litellm_executable), "--port", "4002", "--config", str(config_path)]

# "w" crée le journal ou remplace son contenu précédent.
# Le bloc with ferme le fichier côté notebook après le lancement ; le processus
# enfant conserve sa propre référence au fichier pour continuer à écrire dedans.
with log_path.open("w") as log_file:
    # Popen lance le programme et rend immédiatement la main au notebook.
    # Il crée un processus séparé : ce n'est pas un thread Python.
    proxy_process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,  # Dossier de travail du serveur ; contient notamment .env.
        env=proxy_env,     # Variables d'environnement transmises au serveur.
        stdout=log_file,   # Écrire les messages ordinaires dans proxy.log.
        stderr=subprocess.STDOUT,  # Écrire aussi les erreurs dans le même journal.
    )

proxy_processes["llm-lanfuse.yaml"] = proxy_process

# Le lancement d'un processus ne garantit pas que le serveur est déjà prêt.
# Faire au maximum 60 essais ; chacun peut prendre jusqu'à 2 secondes pour
# la requête HTTP, puis 1 seconde de pause. Ce n'est donc pas un délai strict de 60 s.
for attempt in range(60):
    # Si le programme s'est arrêté, afficher la fin de son journal pour diagnostiquer.
    # [-6000:] conserve les 6000 derniers caractères, pas les 6000 dernières lignes.
    if proxy_process.poll() is not None:
        raise RuntimeError(log_path.read_text()[-6000:])

    try:
        # GET /v1/models demande les modèles exposés par le proxy.
        # Cette vérification ne génère pas de réponse avec un modèle fournisseur.
        status = httpx.get(
            "http://127.0.0.1:4002/v1/models",
            # Bearer est le schéma d'authentification ; la valeur est la clé maître.
            headers={"Authorization": "Bearer " + os.environ["LITELLM_MASTER_KEY"]},
            timeout=2,  # Délai des opérations réseau de cette vérification.
        )
        # HTTP 200 signifie que cette route répond avec succès et accepte la clé.
        # Cela ne prouve pas encore que chaque modèle fournisseur est utilisable.
        if status.status_code == 200:
            print("Proxy prêt sur http://127.0.0.1:4002/v1")
            break  # Quitter la boucle dès que la vérification réussit.

    except httpx.RequestError:
        # Pendant le démarrage, le port peut ne pas encore accepter de connexion.
        # Ignorer cet échec réseau provisoire et réessayer après la pause.
        # Une réponse HTTP 401/500 ne passe pas ici : elle échoue au test du code 200.
        pass

    time.sleep(1)  # Éviter d'envoyer les vérifications en continu.

else:
    # Le else appartient au for : il s'exécute si les 60 essais se terminent sans break.
    # Le processus peut encore fonctionner ; inspecter son journal avant de réessayer.
    raise RuntimeError("Proxy non disponible. Consulte logs/observation-proxy.log.")


In [ ]:
# Client dédié : les autres sections conservent leurs propres ports.
import openai
from pprint import pprint
observation_client = openai.OpenAI(
    api_key=os.environ["LITELLM_MASTER_KEY"],
    base_url="http://127.0.0.1:4002/v1",
    max_retries=0,
)


In [ ]:
messages = [
    {
        "role": "user", # system, assistant
        "content": "لماذا تبدو السماء زرقاء بالنهار؟"
    }
]

response = observation_client.chat.completions.create(
    model="openai-gpt4o-mini",
    messages=messages,
)

In [ ]:
pprint(
    response.choices[0].message.content
)

### LiteLLm + LangChain

In [ ]:
!pip install -qU langchain-openai langchain langchain_community

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader([
                            "https://lilianweng.github.io/posts/2023-06-23-agent/",
                            "https://lilianweng.github.io/posts/2024-07-07-hallucination/",
                            "https://lilianweng.github.io/posts/2024-02-05-human-data-quality/",
                        ])
docs = loader.load()

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    api_key=os.environ["LITELLM_MASTER_KEY"],
    openai_api_base="http://127.0.0.1:4000/v1",
    model = "openai-gpt4o-mini",
    temperature=0.1
)

In [ ]:
map_prompt = ChatPromptTemplate.from_messages(
    [("system", "Write a concise summary of the following:\\n\\n{context}")]
)

map_chain = map_prompt | llm | StrOutputParser()

In [ ]:
# Invoke chain
result = map_chain.invoke({"context": docs})

In [ ]:
result